# Prep 5 · A learned prior: the whole project in one dimension

**Time:** about 90 minutes. **Needs:** Prep 4. Runs on a laptop CPU in a couple of minutes.

In Prep 4 the prior was a matrix of bumps you chose by hand. Real anatomy is not "a sum of bumps", and
nobody can write down what a knee looks like. So we **learn** the prior from examples with a
**variational autoencoder (VAE)**: an *encoder* compresses each example to a few latent numbers, a
*decoder* expands latent numbers back to an example, and after training the decoder alone is our prior:

$$z \sim N(0, I), \qquad x = \text{decode}(z).$$

We do it in 1-D on synthetic signals so it trains in a minute, then plug the trained decoder into the
NumPyro model from Prep 4. That is, line for line, what the school does with knees.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib.pyplot as plt
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

key = jax.random.PRNGKey(0)
n = 32                        # signal length

## 1. The data: a family of smooth signals

Two or three bumps of random position, width and height, normalised to `[0, 1]`. Think of it as the
population of "knees" the prior must learn.

In [ ]:
def make_signals(count, seed=0):
    rng = np.random.default_rng(seed)
    pos = np.arange(n)
    out = np.zeros((count, n), np.float32)
    for i in range(count):
        s = np.zeros(n)
        for _ in range(rng.integers(2, 4)):
            s += rng.uniform(0.4, 1.0) * np.exp(-0.5 * ((pos - rng.uniform(4, n - 4)) / rng.uniform(1.5, 4.0)) ** 2)
        out[i] = s / s.max()
    return out

train = jnp.asarray(make_signals(4000, seed=0))
test = jnp.asarray(make_signals(8, seed=123))         # held out - never seen in training
plt.plot(np.asarray(train[:6]).T); plt.title("six training signals"); plt.show()

## 2. The VAE, and the one line you write

Encoder: signal → `(μ, log σ²)` of a Gaussian over `z`. Decoder: `z` → signal. Training maximises the
**ELBO**: reconstruct well (mean squared error) while keeping the latent Gaussian close to `N(0, I)`
(a KL term, weighted by β).

There is a catch: to reconstruct, we must *sample* `z ~ N(μ, σ²)` — and you cannot differentiate through
a random draw. The **reparameterisation trick** fixes it: draw `ε ~ N(0, I)` once, then set
`z = μ + σ · ε` with `σ = exp(½ log σ²)`. Now `z` is a deterministic function of `μ` and `σ`, gradients flow,
and the randomness lives in `ε`.

### Exercise 1 — implement `reparameterise` in the repo

Open `src/mrigen/models/vae.py` and fill in `reparameterise(mu, logvar, key)` — it is one line. The
same function will be used by the school's 2-D VAE; here we use it for the 1-D one. The check draws
5000 samples and verifies their mean and standard deviation.

In [ ]:
import importlib
import mrigen.models.vae
importlib.reload(mrigen.models.vae)
from mrigen.models.vae import reparameterise

mu, logvar = jnp.array([1.0, -2.0]), jnp.log(jnp.array([0.25, 4.0]))
try:
    zs = jax.vmap(lambda k: reparameterise(mu, logvar, k))(jax.random.split(key, 5000))
    assert zs.shape == (5000, 2)
    assert bool(jnp.allclose(zs.mean(0), mu, atol=0.1)), "sample mean should be mu"
    assert bool(jnp.allclose(zs.std(0), jnp.array([0.5, 2.0]), atol=0.1)), "sample std should be exp(0.5 * logvar)"
    print("reparameterise OK - the VAE below can train")
except NotImplementedError as e:
    print("Not yet:", e)
    print("-> implement reparameterise in src/mrigen/models/vae.py, then re-run this cell")

In [ ]:
class VAE1D(eqx.Module):
    """A tiny MLP VAE for length-n signals. The 2-D one in the repo is the same shape with convolutions."""
    enc: eqx.nn.MLP
    dec: eqx.nn.MLP
    latent: int = eqx.field(static=True)

    def __init__(self, latent=4, width=64, *, key):
        k1, k2 = jax.random.split(key)
        self.enc = eqx.nn.MLP(n, 2 * latent, width, depth=2, key=k1)
        self.dec = eqx.nn.MLP(latent, n, width, depth=2, key=k2)
        self.latent = latent

    def encode(self, x):
        h = self.enc(x)
        return h[: self.latent], h[self.latent :]          # mu, logvar

    def decode(self, z):
        return jax.nn.sigmoid(self.dec(z))                  # signals live in [0, 1]


def vae_loss(model, x, key, beta):
    mu, logvar = model.encode(x)
    z = reparameterise(mu, logvar, key)                   # <- your function
    x_hat = model.decode(z)
    recon = jnp.sum((x_hat - x) ** 2)                     # reconstruction error
    kl = -0.5 * jnp.sum(1.0 + logvar - mu ** 2 - jnp.exp(logvar))   # KL[N(mu, s^2) || N(0, 1)]
    return recon + beta * kl


@eqx.filter_jit
def train_step(model, opt_state, batch, key, beta):
    def batched(m):
        keys = jax.random.split(key, batch.shape[0])
        return jnp.mean(jax.vmap(lambda x, k: vae_loss(m, x, k, beta))(batch, keys))
    loss, grads = eqx.filter_value_and_grad(batched)(model)
    updates, opt_state = optim.update(grads, opt_state, eqx.filter(model, eqx.is_array))
    return eqx.apply_updates(model, updates), opt_state, loss


key, mk = jax.random.split(key)
vae = VAE1D(latent=4, key=mk)
optim = optax.adam(2e-3)
opt_state = optim.init(eqx.filter(vae, eqx.is_array))
beta, batch_size, steps = 0.05, 128, 3000

t0 = time.perf_counter()
for step in range(steps):
    key, bk, sk = jax.random.split(key, 3)
    batch = train[jax.random.choice(bk, train.shape[0], (batch_size,), replace=False)]
    vae, opt_state, loss = train_step(vae, opt_state, batch, sk, beta)
    if step % 500 == 0 or step == steps - 1:
        print(f"step {step:4d}  loss {float(loss):.3f}")
print(f"trained in {time.perf_counter() - t0:.0f}s")

## 3. What did it learn? Sample the prior

Draw `z ~ N(0, I)`, decode. If training worked, every sample is a plausible member of the family — smooth
bumps, never noise — even though no sample is a training signal. Then reconstruct a held-out signal
through the bottleneck: four numbers are enough to describe it.

In [ ]:
key, zk = jax.random.split(key)
samples = jax.vmap(vae.decode)(jax.random.normal(zk, (6, vae.latent)))
recons = jax.vmap(lambda x: vae.decode(vae.encode(x)[0]))(test)     # decode the mean latent (no noise)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(np.asarray(samples).T); ax[0].set_title("six samples from the learned prior")
ax[1].plot(np.asarray(test[:3]).T, "k", alpha=.5); ax[1].plot(np.asarray(recons[:3]).T, "--"); ax[1].set_title("held-out signals (black) and their 4-number reconstructions (dashed)")
plt.tight_layout(); plt.show()

recon_err = float(jnp.mean((recons - test) ** 2))
print("held-out reconstruction MSE:", round(recon_err, 4))
assert recon_err < 0.01, "the VAE should reconstruct held-out signals well - if not, train longer"

## 4. Plug the decoder into the reconstruction

NumPyro needs `decode` to be a **pure function** of `z` with the trained weights baked in. Equinox makes
this a two-liner: split the model into its arrays and its structure, and recombine inside a closure.
(The repo's `make_decoder_fn` does exactly this — CLAUDE.md, Contract 4.)

Then the measurement, exactly as in Prep 4: a held-out signal, its Fourier transform, a mask keeping the
centre plus every 4th frequency, noise.

In [ ]:
params, static = eqx.partition(vae, eqx.is_array)
def decode(z):
    return eqx.combine(params, static).decode(z)          # pure: closes over arrays only

def fft1c(x):
    return jnp.fft.fftshift(jnp.fft.fft(jnp.fft.ifftshift(x), norm="ortho"))
def ifft1c(k):
    return jnp.fft.fftshift(jnp.fft.ifft(jnp.fft.ifftshift(k), norm="ortho"))

x_true = test[0]
mask = np.zeros(n, np.float32); mask[::4] = 1; mask[n//2 - 2:n//2 + 2] = 1
mask = jnp.asarray(mask)
sigma = 0.03
key, k1, k2 = jax.random.split(key, 3)
y = mask * fft1c(x_true) + mask * sigma * (jax.random.normal(k1, (n,)) + 1j * jax.random.normal(k2, (n,)))
x_zf = ifft1c(y).real
print(f"measured {int(mask.sum())} of {n} frequencies")

### Exercise 2 — the reconstruction model with a learned prior

This is `recon_model` from the repo, in 1-D. Same five lines as Prep 4's `recon_1d`, with `decode(z)`
in place of `A @ z` and `latent` in place of `A.shape[1]`.

In [ ]:
def recon_learned(y_obs, mask, decode, latent, sigma):
    # YOUR CODE HERE: prior on z, x = decode(z), k = mask * fft1c(x), then observe y_re and y_im
    ...

In [ ]:
mcmc = MCMC(NUTS(recon_learned), num_warmup=500, num_samples=1000, progress_bar=False)
mcmc.run(key, y, mask, decode, vae.latent, sigma)
assert "z" in mcmc.get_samples(), "your model must have a sample site named 'z'"
mcmc.print_summary()
x_samples = jax.vmap(decode)(mcmc.get_samples()["z"])
x_mean, x_std = x_samples.mean(0), x_samples.std(0)

pos = np.arange(n)
err = lambda est: float(jnp.linalg.norm(est - x_true) / jnp.linalg.norm(x_true))
plt.fill_between(pos, x_mean - 2 * x_std, x_mean + 2 * x_std, alpha=.25, label="posterior ± 2 std (uncertainty)")
plt.plot(x_true, "k", label="truth"); plt.plot(x_zf, alpha=.6, label=f"zero-filled (err {err(x_zf):.2f})"); plt.plot(x_mean, label=f"posterior mean (err {err(x_mean):.2f})")
plt.legend(); plt.title("1-D MRI with a learned prior"); plt.show()

# check
assert err(x_mean) < err(x_zf), "the learned prior should beat zero-filling"
assert err(x_mean) < 0.25
print("exercise 2 OK - you have reconstructed a signal from a third of its Fourier coefficients, with error bars")

## 5. Is the uncertainty honest?

A cheap calibration check, the same one the evaluation thread will do on knees: where the posterior std is large, is the
actual error also large? Sort the samples by predicted std, bin them, and compare.

In [ ]:
abs_err = np.abs(np.asarray(x_mean - x_true)); std = np.asarray(x_std)
order = np.argsort(std); bins = np.array_split(order, 4)
for i, b in enumerate(bins):
    print(f"std quartile {i + 1}: predicted std {std[b].mean():.3f}   actual |error| {abs_err[b].mean():.3f}")

**What you will probably see:** the predicted std is several times *smaller* than the actual error, and the
error barely grows with it. The posterior is **overconfident**. The decoder can only produce signals it
learned to produce; the held-out signal is not exactly one of them, and the model has no way to say "my
prior cannot represent this" — so it reports tight error bars around the nearest signal it *can* make.
This is *model misspecification*, and it is the most important thing to know about uncertainty from a
learned prior: **the error bars are only as honest as the model.**

Try it: set `sigma = 0.1` in the measurement cell and re-run from there. The band widens and the
quartiles line up better, because the larger noise term is absorbing the decoder's error. At the school,
the evaluation thread's calibration curve is exactly this check on knees — and the β-VAE's blurriness will show up in it.

## What the school adds

| here | at the school |
|---|---|
| 32 samples, 4 latent numbers | 128 × 128 pixels, 128 latent numbers |
| an MLP trained in a minute | a convolutional β-VAE (a pre-trained checkpoint is provided) |
| bumps | real knees from fastMRI |
| `fft1c` | `fft2c` — everything else in the model is identical |
| four quartiles | a per-pixel uncertainty map and a calibration curve |

## Optional — rung 6, if you're ahead

Open `src/mrigen/recon/vae_numpyro.py` and write `recon_model`: it is `recon_learned` with `fft2c`.
Then `pixi run slice` trains a tiny 2-D VAE on synthetic phantoms and reconstructs one with MAP —
the whole pipeline, CPU only. If you get here before October, Monday afternoon is going to be fun.

## Done when

- `reparameterise` passes (and `pixi run check` now reports a VAE loss instead of "not runnable yet");
- you have the plot with the uncertainty band, and the learned prior beat zero-filling;
- you can say why the reparameterisation trick is needed and why `decode` must be a pure function.

**You have now done the school project in one dimension.** See you in Cape Town.